---
title: Linked-Read Metrics
subtitle: Linked-read metrics relating to barcodes and molecules
date: "9999-12-31"
edit_url: null
---

**Linked-read type**: PLACEHOLDER

In [ ]:
import altair as alt
from harpy.report.components import colored_boxes, print_html, standard_itable
from harpy.report.utilities import binned_histogram, nxx, last_line
from harpy.report.theme import palette
import itables
import os
import numpy as np
import pandas as pd
from pathlib import Path
import sys
itables.init_notebook_mode(connected=True)


In [ ]:
indir = "reports/data/bxstats/"
platform = "haplotagging"


In [ ]:
def process_input(infile: str) -> dict:
    samplename = os.path.basename(infile).replace(".bxstats.gz", "")

    tb = pd.read_table(infile, sep = '\t').drop(columns = ["start", "end"])
   
    # Filter conditions - create masks once
    is_valid = tb['molecule'] >= 0
    is_multiplex = is_valid & (tb['reads'] >= 2)
    
    # Calculate metrics using masks
    multiplex_df = tb[is_multiplex]
    #singletons = ((tb['reads'] < 2) & is_valid).sum()
    tot_mol = is_valid.sum()
    tot_valid_reads = tb.loc[is_valid, 'reads'].sum()
    tot_invalid_reads = tb.loc[~is_valid, 'reads'].sum()
    tot_reads = tot_valid_reads + tot_invalid_reads
    # Stats on multiplex molecules
    avg_reads_per_mol = multiplex_df['reads'].mean().round(1)
    #sem_reads_per_mol = multiplex_df['reads'].sem().round(2)
    avg_mol_cov = multiplex_df['coverage_inserts'].mean().round(2)
    #sem_mol_cov = multiplex_df['coverage_inserts'].sem().round(4)
    avg_mol_cov_bp = multiplex_df['coverage_bp'].mean().round(2)
    #sem_mol_cov_bp = multiplex_df['coverage_bp'].sem().round(4)
    _lens = multiplex_df['length_inferred'].sort_values(ascending = False)
    # N-statistics
    n50 = nxx(_lens, 50)
    n75 = nxx(_lens, 75)
    n90 = nxx(_lens, 90)
    
    # Read total unique barcodes from footer
    tot_uniq_bx = int(last_line(infile).split(":")[-1])
    
    return {
        "Sample" : samplename,
        "Total Reads" : tot_reads,
        "Valid" : round(tot_valid_reads/ tot_reads, 4),
        "Unique Molecules" : tot_mol,
        "Linked" : is_multiplex.sum() / tot_mol,
        "Barcodes" : tot_uniq_bx,
        "Barcodes/Molecule" : round(tot_mol / tot_uniq_bx,2),
        "Reads/Molecule" : avg_reads_per_mol,
        #"se_readspermol" : sem_reads_per_mol,
        "Insert Molecule Coverage" : avg_mol_cov,
        #"se_molcov" : sem_mol_cov,
        "BP Molecule Coverage" : avg_mol_cov_bp,
        #"se_molcovbp" : sem_mol_cov_bp,
        "N50" : n50,
        "N75" : n75,
        "N90" : n90
    }


In [ ]:
agg_cols = ["Sample", "Total Reads", "Valid", "Unique Molecules", "Linked", "Barcodes", "Barcodes/Molecule", "Reads/Molecule", "Insert Molecule Coverage", "BP Molecule Coverage", "N50", "N75", "N90"]
aggregate_df = {}
for i in agg_cols:
    aggregate_df[i] = []

for i in Path(indir).glob("*.bxstats.gz"):
    _sample = process_input(str(i))
    for k,v in _sample.items():
        aggregate_df[k].append(v)

aggregate_df = (
    pd.DataFrame(aggregate_df)
    .sort_values(by = 'Sample')
    .replace(np.nan, 0)
    .reset_index(drop = True)
)


This report aggregates the barcode-specific information from the alignments
that were created using `harpy align`. Detailed information for any one sample
can be found in that sample's individual report.

In [ ]:
if len(aggregate_df) == 0:
    print_html("All input files were empty, no report to show")
    sys.exit(0)


In [ ]:
_n50 = (aggregate_df['N50'].replace(0, np.nan).mean()/1000).round()
_n75 = (aggregate_df['N75'].replace(0, np.nan).mean()/1000).round()
_n90 = (aggregate_df['N90'].replace(0, np.nan).mean()/1000).round()

(
    colored_boxes()
    .add(len(aggregate_df), "Samples")
    .conditional(aggregate_df['Linked'].mean().round(3), "Avg Linked", 0.4, as_percent = True)
    .conditional(aggregate_df['Valid'].replace(0, np.nan).mean(), "Avg Valid BX", 0.4, as_percent=True)
    .conditional(aggregate_df['Reads/Molecule'].replace(0, np.nan).mean(), "Avg Reads/Mol", 2)
    .add(_n50, "N50 (kb)", width = 220)
    .add(_n75, "N75 (kb)", width = 220)
    .add(_n90, "N90 (kb)", width = 220)
).render()


## Sample Information

The chart below is visual representation of some of the common linked-read  metrics that may help assess the success of sample/library preparation. For more details, see the table beneath the visualization and for even more granularity, you can inspect the report generated for each individual sample. 

In [ ]:
(
    alt.Chart(aggregate_df[['Sample', 'Valid', 'Linked', 'Insert Molecule Coverage', 'BP Molecule Coverage']])
    .transform_fold(['Valid', 'Linked', 'Insert Molecule Coverage', 'BP Molecule Coverage'])
    .transform_calculate(random="random()")
    .mark_point(size = 65)
    .encode(
        x = alt.X('value:Q', title = 'Percent').scale(domain = [0,1]).axis(format = "%"),
        y = alt.Y('key:N', title = None),
        yOffset="random:Q",
        color = 'key:N',
        shape = 'key:N',
        tooltip = [
            'Sample:N',
            alt.Tooltip('key:N', title = "Metric"),
            alt.Tooltip('value:Q', title = "Percent", format = '.2%')
            ]
    )
    .configure_legend(orient = "top")
    .properties(
        title=alt.Title('Per-Sample Metrics'),
        usermeta={'embedOptions': {'downloadFileName': f'alignments.per.sample'}}   
    )
)


This table [^1] is an aggregation of data for each sample based on their `*.bxstats.gz` file.
Every column after `Unique Molecules` ignores singletons in its calculations.

[^1]:
    | Column | Description |
    |:-------|:------------|
    | `Sample` | name of the sample |
    | `Total Reads`| total number of alignments |
    | `Valid` | proportion of valid barcoded alignments |
    | `Unique Molecules` | the unique DNA molecules as inferred from linked-read barcodes |
    | `Linked` | molecules composed of two or more single/paired-end sequences, in other words, molecules with linked-read information |
    | `Barcodes` | number of unique barcodes, which may differ from unique molecules after deconvolution |
    | `Barcodes/Molecule` | molecule-to-barcode ratio, which helps benchmark deconvolution performance, if performed |
    | `Reads/Molecule` | average number of reads per unique molecule |
    | `Insert Molecule Coverage` | average percent of a molecule that is covered by a read, where coverage includes unsequenced gaps between linked reads |
    | `BP Molecule Coverage` | average percent molecule coverage, where coverage only includes sequences and not the gaps between linked reads |
    | `N50` | N50 of inferred molecules |
    | `N75` | N75 of inferred molecules |
    | `N90` | N90 of inferred molecules |

In [ ]:
aggregate_df.insert(1,"Report", 
    [f'<a href="./{i}">link</a>' for i in aggregate_df['Sample']]
)
standard_itable(aggregate_df, "alignment.bx", fixedcols=2, html = True)


## Library Performance
### NX metrics

The **NX** metrics (e.g. **N50**) tend to be useful for understanding centrality for molecule length in genomic contexts.
With your linked-read sequences aligned to a reference genome, you can make inferences about the lengths of the original molecules from which linked-read fragments were derived. These are the distributions of three common NX metrics (N50, N75, N90) across all samples are given below.

In [ ]:
selection = alt.selection_point(fields=['Nxx'], bind='legend')

(
    alt.Chart(aggregate_df[['Sample', 'N50', 'N75', 'N90']])
    .transform_fold(
        ['N50', 'N75', 'N90'],
        as_=['Nxx', 'value']
    )
    .transform_density(
        'value',
        groupby=['Nxx'],
        as_=['value', 'density']
    )
    .mark_area(interpolate = "basis")
    .encode(
        x=alt.X('value:Q', bin = "binned").axis(title='Length (kb)', labelExpr='datum.value / 1000'),
        y=alt.Y('density:Q', title='Density').axis(labels = False),
        color=alt.Color('Nxx:N', title='Metric').scale(domain = ['N50','N75','N90'])
    )
    .transform_filter(selection)
    .add_params(selection)
    .configure_legend(orient = "top")
    .properties(
        title=alt.Title('Nxx Values', subtitle = 'Values derived using non-singleton molecules'),
        usermeta={'embedOptions': {'downloadFileName': f'alignments.nxx'}}
    )
)


### Reads per molecule
This distribution shows the average number of reads per inferred molecule across your samples.

In [ ]:
_hist = binned_histogram(aggregate_df['Reads/Molecule'], 0.25, True)
(
    alt.Chart(_hist)
    .mark_area(interpolate = "monotone")
    .encode(
        x=alt.X('interval:O', bin = 'binned').axis(title='Reads Per Molecule', tickMinStep=0.5, labelAngle=-40),
        y=alt.Y('proportion:Q', title='Percent of Samples').axis(format = '%'),
        tooltip = [
            alt.Tooltip('interval', title = "Reads Per Molecule", bin = "binned"),
            alt.Tooltip('proportion', title = "Percent of Samples").format('.1%')
        ]
    )
    .properties(
        title=alt.Title('Reads Per Molecule', subtitle = 'Values derived using non-singleton molecules'),
        usermeta={'embedOptions': {'downloadFileName': f'alignments.readspermol'}}
    )
)


### Valid barcodes
This is a distribution of what percent of total alignments each sample had valid barcodes[^3]:

[^3]:
    Valid barcodes with respect to the conventions of a given linked-read chemistry:
    
    **haplotagging**: `AxxCxxBxxDxx` where `xx` is not `00`
    
    **stLFR**: `X_Y_Z` where `X`, `Y`, or `Z` is not `0`
    
    **TELLseq**: there is no `N` nucleotide


In [ ]:
_hist = binned_histogram(aggregate_df['Valid'], 0.1, True, 1)
(
    alt.Chart(_hist)
    .mark_area(interpolate = "monotone")
    .transform_filter(alt.datum.bin <= 1)
    .encode(
        x=alt.X('interval:O').axis(title='Proportion Valid Barcodes', labelAngle=-40),
        y=alt.Y('proportion:Q', title='Percent of Samples').axis(format = '%').scale(domain=[0,1]),
        tooltip = [
            alt.Tooltip('interval', title = "Proportion Valid Barcodes", bin = "binned"),
            alt.Tooltip('proportion', title = "Percent of Samples").format('.1%')
        ]
    )
    .properties(
            title=alt.Title('Proportion Valid Barcodes'),
        usermeta={'embedOptions': {'downloadFileName': f'alignments.percent.valid'}}
    )
)


### Linking proportion
This scatterplot is a diagnostic that shows the relationship between the proportion of linked
reads (reads with linked information, aka not singletons) compared to total reads. Each point is a sample and is colored by the
mean number of reads per molecule for that sample.

In [ ]:
_hist = binned_histogram(aggregate_df['Linked'], 0.1, True, 1, )
(
    alt.Chart(_hist)
    .mark_area(interpolate = "monotone")
    .transform_filter(alt.datum.bin <= 1)
    .encode(
        x=alt.X('interval:O', bin = 'binned').axis(title='Proportion Linked Reads', tickMinStep=0.1, labelAngle=-40),
        y=alt.Y('proportion:Q', title='Percent of Samples').axis(format = '%').scale(domain=[0,1]),
        tooltip = [
            alt.Tooltip('interval', title = "Proportion Linked Reads", bin="binned"),
            alt.Tooltip('proportion', title = "Percent of Samples").format('.1%')
        ]
    )
    .properties(
        title=alt.Title('Proportion Linked Reads'),
        usermeta={'embedOptions': {'downloadFileName': f'alignments.percent.linked'}}
    )
)


## Supporting Info

:::{dropdown} NX values
`NX` is the length of the shortest molecule that when you sum the lengths of every molecule larger than it, represents at least **X%** of the total molecules by length. For example, `N50` would be the molecule length at which the sum of all the molecule lengths larger than it would
represent **50%** of the total molecules by length (sort of like a cumulative median).

As an example, imagine we had molecules with lengths [4, 3, 2, 1], making up a total length of 10. The `N50`
would be the first number (starting from the biggest) that sums up to at least 5 (50% of 10), which is `3`, because `4` + `3` = 7. The `N90` would be the first number that sums up to at least 9 (90% of 10), which is `2` because `4` + `3` + `2` = 9.
:::

:::{dropdown} Understanding inferred molecule lengths
Harpy uses the highest and lowest mapping positions of a read cluster sharing the same barcode (incorporating distance-based deconvolution thresholds, if configured to do so) to infer the original molecule length. Given that it's mostly impossible to understand how much of a source molecule the mapped reads actually covered, it would be more appropriate to think of the inferred molecule lengths (and NX measures) to be more like _"at least this long"_, since the reads likely did not originate from the ends of the source DNA molecule. The absolute length of the source DNA molecule would be a more useful diagnostic in troubleshooting/modifying the actual linked-read chemistry, whereas the the inferred molecule length describes what length of it is actually represented in the sequences and thus more useful in downstream/analytical contexts.
:::